# Fiche 2 — Normaliser : 1NF, 2NF, 3NF

| | |
|---|---|
| 🎯 **Objectif** | Expliquer les trois formes normales avec un exemple, et pourquoi on découpe en plusieurs tables |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 1 |
| 🔗 **Dans le projet** | Partie 1, étape 2 (schéma relationnel) |

> **L'idée clé.** Normaliser, c'est ranger chaque information **à un seul endroit**. Une information répétée finit toujours par se contredire.

In [1]:
import sqlite3

import pandas as pd

# Une base « jetable » en mémoire : on peut tout casser, rien n'est écrit sur le disque.
# isolation_level=None : chaque instruction est validée aussitôt, comme dans SQLite Studio.
base = sqlite3.connect(":memory:", isolation_level=None)


def sql(requete: str) -> pd.DataFrame | None:
    """Exécute du SQL (une ou plusieurs instructions) et affiche le résultat s'il y en a un."""
    if requete.strip().rstrip(";").count(";"):
        base.executescript(requete)
        return None
    curseur = base.execute(requete)
    if curseur.description:
        return pd.DataFrame(curseur.fetchall(), columns=[c[0] for c in curseur.description])
    return None

sql("""
CREATE TABLE ventes_plates (
    id_vente INTEGER PRIMARY KEY,
    adresse  TEXT,
    commune  TEXT,
    region   TEXT
);
INSERT INTO ventes_plates VALUES
    (1, '2 Cour de l''île Louviers, 75004 Paris', 'Paris 4e', 'Ile-de-France'),
    (2, '10 rue de Rivoli, 75004 Paris',            'Paris 4e', 'Ile-de-France'),
    (3, '5 avenue Jean Médecin, 06000 Nice',        'Nice',     'Provence-Alpes-Côte d''Azur');
""")
sql("SELECT * FROM ventes_plates")

,id_vente,adresse,commune,region
0,1,"2 Cour de l'île Louviers, 75004 Paris",Paris 4e,Ile-de-France
1,2,"10 rue de Rivoli, 75004 Paris",Paris 4e,Ile-de-France
2,3,"5 avenue Jean Médecin, 06000 Nice",Nice,Provence-Alpes-Côte d'Azur


## 1. Première forme normale (1NF) : des valeurs atomiques

Une valeur est **atomique** si on ne peut pas la découper sans perdre son sens. `adresse` colle 5 informations
(numéro, type de voie, voie, code postal, ville) : filtrer « les ventes du 75004 » devient du bricolage de texte.

In [2]:
sql("SELECT id_vente FROM ventes_plates WHERE adresse LIKE '%75004%'")  # fragile : dépend de l'écriture

,id_vente
0,1
1,2


En 1NF, l'adresse devient `no_voie`, `type_voie`, `voie`, `code_postal`. « Cour de l'île Louviers » reste en un seul
morceau : la découper en mots ne voudrait plus rien dire.

## 2. Deuxième forme normale (2NF) : tout dépend de **toute** la clé

La 2NF concerne les clés composées de plusieurs colonnes. Imaginons une table des **lots** vendus (un lot = un
appartement, une cave, un parking…), avec pour clé le couple (vente, numéro de lot) :

In [3]:
sql("""
CREATE TABLE lot (
    id_vente    INTEGER,
    numero_lot  INTEGER,
    surface_lot REAL,
    date_vente  TEXT,                       -- dépend seulement de id_vente : violation de la 2NF
    PRIMARY KEY (id_vente, numero_lot)
);
INSERT INTO lot VALUES (1, 1, 48.2, '2020-01-02'), (1, 2, 6.0, '2020-01-02'), (2, 1, 71.5, '2020-01-03');
""")
sql("SELECT * FROM lot")

,id_vente,numero_lot,surface_lot,date_vente
0,1,1,48.2,2020-01-02
1,1,2,6.0,2020-01-02
2,2,1,71.5,2020-01-03


La surface dépend du lot (de toute la clé) ; la date ne dépend que de la vente (d'une **partie** de la clé) : elle
est recopiée pour chaque lot de la même vente. En 2NF, la date part dans la table `vente`.

## 3. Troisième forme normale (3NF) : ⚠️ le piège de la redondance

`region` dépend de `commune`, qui n'est pas la clé : c'est une **dépendance transitive** (vente → commune → région).
Conséquence, si on corrige l'orthographe sur une seule ligne :

In [4]:
sql("UPDATE ventes_plates SET region = 'Île-de-France' WHERE id_vente = 1")
sql("SELECT region, COUNT(*) AS nb_ventes FROM ventes_plates GROUP BY region")

,region,nb_ventes
0,Ile-de-France,1
1,Provence-Alpes-Côte d'Azur,1
2,Île-de-France,1


À coller dans SQLite Studio :
```sql
UPDATE ventes_plates SET region = 'Île-de-France' WHERE id_vente = 1;
SELECT region, COUNT(*) AS nb_ventes FROM ventes_plates GROUP BY region;
```

La même région existe maintenant sous **deux noms** : c'est une **anomalie de mise à jour**. La solution : une table
par entité, reliées par des clés ; le nom de la région n'est plus stocké qu'une fois.

In [5]:
sql("""
CREATE TABLE region  (id_region TEXT PRIMARY KEY, nom_region TEXT NOT NULL);
CREATE TABLE commune (id_commune TEXT PRIMARY KEY, nom_commune TEXT NOT NULL,
                      id_region TEXT NOT NULL REFERENCES region (id_region));
INSERT INTO region  VALUES ('11', 'Ile-de-France'), ('93', 'Provence-Alpes-Côte d''Azur');
INSERT INTO commune VALUES ('75104', 'Paris 4e', '11'), ('06088', 'Nice', '93');
UPDATE region SET nom_region = 'Île-de-France' WHERE id_region = '11';
""")
sql("SELECT c.nom_commune, r.nom_region FROM commune AS c INNER JOIN region AS r ON r.id_region = c.id_region")

,nom_commune,nom_region
0,Paris 4e,Île-de-France
1,Nice,Provence-Alpes-Côte d'Azur


Une seule ligne modifiée, toutes les communes en profitent : impossible de se contredire. (La jointure `INNER JOIN … ON` est détaillée en fiche 7.)

### ✍️ À toi de jouer

Dans le schéma du projet, la table `commune` contient `id_region` mais pas `nom_region`. Explique en 2 phrases pourquoi ajouter `nom_region` dans `commune` violerait la 3NF, et quelle anomalie cela provoquerait.

**Ta réponse** (quelques phrases) : …

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```text
nom_region dépend de id_region, qui n'est pas la clé de commune : dépendance transitive (commune → région → nom).
Le nom serait répété pour chaque commune de la région ; une correction oubliée créerait deux orthographes.
```


</details>

## ✅ À retenir

1. **1NF** : une valeur atomique par cellule (l'adresse est découpée, la rue reste entière).
2. **2NF** : avec une clé composée, chaque colonne dépend de **toute** la clé.
3. **3NF** : aucune colonne ne dépend d'une colonne non-clé ; chaque information à un seul endroit.
4. La redondance ne coûte pas que de la place : elle crée des **contradictions**.